# Vehicle Route Optimization Using Ant Colony Optimization

**Experiment 9 | Nature-Inspired Computing mini project | Project 5**  
**Team:** Hardik Hinduja (20), Avinash Gehi (11), Sahil Deshmukh (9), Siddharth Dey (10)

This notebook compares Ant Colony Optimization (ACO) with a 500-tour random baseline, nearest neighbor, and exhaustive enumeration on a small ten-location delivery loop. ACO uses pheromone trails and inverse-distance desirability, followed by deterministic 2-opt local improvement.

> All coordinates are synthetic and illustrative. Distances are Euclidean kilometres on a toy plane; they are not road distances, GPS coordinates, or live navigation advice.

## Selected topic
**Project 5: Vehicle Route Optimization Using Ant Colony Optimization (ACO).** Find a short closed route that starts at a depot, visits every delivery stop exactly once, and returns to the depot.

## Guardrails
- Use only the included synthetic coordinates; do not enter addresses, customer names, phone numbers, or live tracking data.
- Every route must start and end at the depot and visit each stop exactly once.
- Recompute the final distance independently by summing the leg matrix, and verify it agrees with the optimizer.
- Compare ACO with exhaustive enumeration on this small instance; on larger instances exact enumeration is not practical.
- A synthetic Euclidean result is not a real-world road route or a proof of traffic-aware optimality.

## Implementation prompt
“Build a complete, reproducible Experiment 9 mini project for Project 5, Vehicle Route Optimization Using Ant Colony Optimization. Use ten clearly synthetic coordinates (one depot and nine delivery stops); calculate a symmetric Euclidean distance matrix; implement seeded ACO with roulette-wheel transitions, pheromone evaporation/deposit, and deterministic 2-opt improvement; compare against 500 random tours, nearest neighbor, and an exact brute-force optimum; print detailed plain tables for parameters, benchmark results, route legs and cumulative distance, locations, pairwise distances, convergence, and independent route audits; export CSVs; explain limitations and guardrails; and provide a responsive white-and-black website with an interactive 3D route view. Never represent toy coordinates as actual addresses or road navigation.”


## 1. Imports and display setup
The optimizer uses the Python standard library. Pandas is used for plain tables and CSV export; matplotlib is optional for the monochrome convergence figure.

In [ ]:
import pandas as pd
from IPython.display import display
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)


## 2. Complete ACO implementation
The following source is embedded from `scripts/route_optimizer.py`, so this notebook can run independently in Colab without cloning the repository.

In [ ]:
"""Synthetic delivery-route benchmark solved with Ant Colony Optimization."""
from __future__ import annotations
import itertools
import math
import random
import time

LOCATIONS = [
    {"id": "DEPOT", "name": "VESIT Depot", "x": 0.0, "y": 0.0},
    {"id": "STOP-01", "name": "Harbour Point", "x": 2.4, "y": 1.8},
    {"id": "STOP-02", "name": "Cedar Market", "x": 4.7, "y": 0.9},
    {"id": "STOP-03", "name": "North Arcade", "x": 5.6, "y": 3.5},
    {"id": "STOP-04", "name": "Lakeview Hub", "x": 3.9, "y": 5.2},
    {"id": "STOP-05", "name": "Garden Square", "x": 1.1, "y": 4.8},
    {"id": "STOP-06", "name": "East Works", "x": 7.4, "y": 5.9},
    {"id": "STOP-07", "name": "Riverside Depot", "x": 8.2, "y": 2.6},
    {"id": "STOP-08", "name": "Hillcrest Store", "x": 6.3, "y": -1.4},
    {"id": "STOP-09", "name": "South Exchange", "x": -1.8, "y": 2.7},
]


def make_distance_matrix(locations=LOCATIONS):
    return [[math.hypot(a["x"] - b["x"], a["y"] - b["y"])
             for b in locations] for a in locations]


def route_distance(route, distances):
    return sum(distances[route[i]][route[i + 1]] for i in range(len(route) - 1))


def valid_route(route, node_count):
    return (len(route) == node_count + 1 and route[0] == 0 and route[-1] == 0
            and sorted(route[1:-1]) == list(range(1, node_count)))


def nearest_neighbor(distances):
    route = [0]
    remaining = set(range(1, len(distances)))
    while remaining:
        current = route[-1]
        nxt = min(remaining, key=lambda node: (distances[current][node], node))
        route.append(nxt)
        remaining.remove(nxt)
    route.append(0)
    return {"route": route, "distance": route_distance(route, distances)}


def random_baseline(distances, seed=17, trials=500):
    rng = random.Random(seed)
    nodes = list(range(1, len(distances)))
    candidates = []
    for _ in range(trials):
        rng.shuffle(nodes)
        route = [0, *nodes, 0]
        candidates.append(route_distance(route, distances))
    return {"route": None, "distance": sum(candidates) / len(candidates),
            "best_distance": min(candidates), "trials": trials}


def exact_optimum(distances):
    best_distance = math.inf
    best_route = None
    for order in itertools.permutations(range(1, len(distances))):
        route = (0, *order, 0)
        length = route_distance(route, distances)
        if length < best_distance:
            best_distance, best_route = length, list(route)
    return {"route": best_route, "distance": best_distance}


def two_opt(route, distances):
    best = route[:]
    improved = True
    while improved:
        improved = False
        for left in range(1, len(best) - 2):
            for right in range(left + 1, len(best) - 1):
                candidate = best[:left] + list(reversed(best[left:right + 1])) + best[right + 1:]
                if route_distance(candidate, distances) + 1e-12 < route_distance(best, distances):
                    best, improved = candidate, True
    return best


def ant_colony(distances, seed=141, ants=48, iterations=160,
               alpha=1.0, beta=4.0, evaporation=0.42, deposit=100.0):
    """ACO with roulette-wheel transitions, elitist pheromone and 2-opt."""
    rng = random.Random(seed)
    n = len(distances)
    pheromone = [[1.0 for _ in range(n)] for _ in range(n)]
    eta = [[0.0 if i == j else 1.0 / max(distances[i][j], 1e-9)
            for j in range(n)] for i in range(n)]
    best_route = None
    best_length = math.inf
    history = []
    for _ in range(iterations):
        iteration_best_route, iteration_best_length = None, math.inf
        for _ant in range(ants):
            route, remaining = [0], set(range(1, n))
            while remaining:
                current = route[-1]
                options = sorted(remaining)
                weights = [(pheromone[current][node] ** alpha) * (eta[current][node] ** beta)
                           for node in options]
                pick = rng.random() * sum(weights)
                chosen = options[-1]
                for node, weight in zip(options, weights):
                    pick -= weight
                    if pick <= 0:
                        chosen = node
                        break
                route.append(chosen)
                remaining.remove(chosen)
            route.append(0)
            route = two_opt(route, distances)
            length = route_distance(route, distances)
            if length < iteration_best_length:
                iteration_best_route, iteration_best_length = route, length
            if length < best_length:
                best_route, best_length = route[:], length
        for i in range(n):
            for j in range(n):
                pheromone[i][j] = max(1e-8, pheromone[i][j] * (1 - evaporation))
        for route, length, amount in [(iteration_best_route, iteration_best_length, 1.0),
                                      (best_route, best_length, 2.0)]:
            for left, right in zip(route, route[1:]):
                pheromone[left][right] += amount * deposit / length
                pheromone[right][left] += amount * deposit / length
        history.append(best_length)
    return {"route": best_route, "distance": best_length, "history": history,
            "ants": ants, "iterations": iterations, "alpha": alpha, "beta": beta,
            "evaporation": evaporation, "seed": seed}


def independent_audit(route, distances, node_count):
    valid = valid_route(route, node_count)
    measured = route_distance(route, distances) if valid else math.inf
    return {"closed_route_starts_and_ends_at_depot": valid and route[0] == route[-1] == 0,
            "each_delivery_stop_visited_once": valid and len(set(route[1:-1])) == node_count - 1,
            "all_nodes_included": valid and set(route[1:-1]) == set(range(1, node_count)),
            "distance_recomputed_from_legs": measured,
            "route_is_valid": valid}


def run_experiment():
    distances = make_distance_matrix()
    timings = {}
    start = time.perf_counter(); random_result = random_baseline(distances); timings["Random (500 tours)"] = time.perf_counter() - start
    start = time.perf_counter(); nearest = nearest_neighbor(distances); timings["Nearest neighbor"] = time.perf_counter() - start
    start = time.perf_counter(); aco = ant_colony(distances); timings["ACO + 2-opt"] = time.perf_counter() - start
    start = time.perf_counter(); exact = exact_optimum(distances); timings["Exact enumeration"] = time.perf_counter() - start
    results = {"Random (500 tours)": random_result, "Nearest neighbor": nearest,
               "ACO + 2-opt": aco, "Exact enumeration": exact}
    for name, result in results.items():
        if name == "Random (500 tours)":
            result["runtime_seconds"] = timings[name]
        else:
            result["runtime_seconds"] = timings[name]
    audit = independent_audit(aco["route"], distances, len(LOCATIONS))
    audit["distance_matches_optimizer"] = math.isclose(audit["distance_recomputed_from_legs"], aco["distance"], rel_tol=0, abs_tol=1e-9)
    audit["gap_to_exact_percent"] = (aco["distance"] / exact["distance"] - 1) * 100
    return {"locations": LOCATIONS, "distances": distances, "results": results,
            "audit": audit}


if __name__ == "__main__":
    r = run_experiment()
    for name, result in r["results"].items():
        score = result.get("distance", result.get("best_distance"))
        print(f"{name}: {score:.3f} km")
    print("Audit:", r["audit"])


## 3. Instance and parameters
One depot plus nine synthetic stops gives 9! = 362,880 possible stop permutations, small enough for an exact reference optimum. The coordinate plane is a toy model and the straight-line distances are illustrative.

In [ ]:
instance_table = pd.DataFrame([
    {'Parameter':'Locations including depot','Value':len(LOCATIONS),'Meaning':'One depot plus nine delivery stops'},
    {'Parameter':'Delivery stops','Value':len(LOCATIONS)-1,'Meaning':'Each stop must appear once'},
    {'Parameter':'Distance metric','Value':'Euclidean','Meaning':'Toy plane; not road routing'},
    {'Parameter':'Random baseline trials','Value':500,'Meaning':'Mean and best sampled tour'},
    {'Parameter':'ACO seed','Value':141,'Meaning':'Reproducible ant decisions'},
    {'Parameter':'Ants per iteration','Value':48,'Meaning':'Candidate tours constructed each iteration'},
    {'Parameter':'ACO iterations','Value':160,'Meaning':'Pheromone update cycles'},
    {'Parameter':'Alpha (pheromone weight)','Value':1.0,'Meaning':'Influence of learned trails'},
    {'Parameter':'Beta (distance heuristic)','Value':4.0,'Meaning':'Influence of inverse distance'},
    {'Parameter':'Evaporation rate','Value':0.42,'Meaning':'Pheromone fraction removed per iteration'},
    {'Parameter':'Local improvement','Value':'2-opt','Meaning':'Deterministic edge reversal improvement'},
    {'Parameter':'Exact reference','Value':'362,880 tours','Meaning':'9! permutations, one depot fixed'},
])
display(instance_table)
location_table = pd.DataFrame(LOCATIONS)
display(location_table)


## 4. Benchmark comparison
Random assignment is averaged over 500 seeded tours. Nearest neighbor always selects the closest unvisited stop. ACO constructs tours with pheromone-guided roulette selection and applies 2-opt. Exact enumeration supplies the global optimum for this small instance.

In [ ]:
experiment = run_experiment()
results = experiment['results']
random_result = results['Random (500 tours)']
benchmark_table = pd.DataFrame([
    {'Method':'Random (500 tours)','Distance (km)':random_result['distance'], 'Reference / detail':f"mean of {random_result['trials']} seeded tours", 'Runtime seconds':random_result['runtime_seconds']},
    {'Method':'Nearest neighbor','Distance (km)':results['Nearest neighbor']['distance'], 'Reference / detail':'greedy nearest unvisited stop', 'Runtime seconds':results['Nearest neighbor']['runtime_seconds']},
    {'Method':'ACO + 2-opt','Distance (km)':results['ACO + 2-opt']['distance'], 'Reference / detail':'48 ants; 160 iterations; seed 141', 'Runtime seconds':results['ACO + 2-opt']['runtime_seconds']},
    {'Method':'Exact enumeration','Distance (km)':results['Exact enumeration']['distance'], 'Reference / detail':'global optimum over 362,880 tours', 'Runtime seconds':results['Exact enumeration']['runtime_seconds']},
])
benchmark_table['Distance (km)'] = benchmark_table['Distance (km)'].round(3)
benchmark_table['Runtime seconds'] = benchmark_table['Runtime seconds'].round(4)
benchmark_table['Gap to exact (%)'] = ((benchmark_table['Distance (km)'] / results['Exact enumeration']['distance'] - 1) * 100).round(3)
display(benchmark_table)


## 5. Optimized route and leg table
The ordered legs include the return to the depot. Cumulative distance is recalculated from the distance matrix.

In [ ]:
aco_result = results['ACO + 2-opt']
route = aco_result['route']
locations = experiment['locations']
distances = experiment['distances']
leg_rows=[]
cumulative=0.0
for sequence, (left, right) in enumerate(zip(route, route[1:]), start=1):
    leg=distances[left][right]
    cumulative += leg
    leg_rows.append({'Leg':sequence,'From':locations[left]['name'],'To':locations[right]['name'],
                     'From ID':locations[left]['id'],'To ID':locations[right]['id'],
                     'Leg distance (km)':leg,'Cumulative (km)':cumulative})
route_table=pd.DataFrame(leg_rows)
route_table[['Leg distance (km)','Cumulative (km)']] = route_table[['Leg distance (km)','Cumulative (km)']].round(3)
display(route_table)


## 6. Pairwise distance matrix
The matrix is symmetric, with a zero diagonal. Distances are straight-line Euclidean values in the illustrative coordinate plane.

In [ ]:
ids=[location['id'] for location in locations]
distance_matrix_table=pd.DataFrame(distances,index=ids,columns=ids).round(2)
display(distance_matrix_table)


## 7. Convergence and independent audit
The audit validates route closure, stop coverage, unique visits, and an independent leg-sum distance. The exact optimum is used to calculate the gap; zero gap applies only to this synthetic instance.

In [ ]:
history=aco_result['history']
exact_distance=results['Exact enumeration']['distance']
audit=experiment['audit']
convergence_table=pd.DataFrame([
    {'Metric':'Best distance after iteration 1 (km)','Value':round(history[0],6)},
    {'Metric':'Best distance after iteration 160 (km)','Value':round(history[-1],6)},
    {'Metric':'Iterations','Value':len(history)},
    {'Metric':'Best-so-far history is non-increasing','Value':all(history[i] >= history[i+1] for i in range(len(history)-1))},
    {'Metric':'Exact optimum (km)','Value':round(exact_distance,6)},
    {'Metric':'ACO gap to exact (%)','Value':round(audit['gap_to_exact_percent'],6)},
])
display(convergence_table)
audit_table=pd.DataFrame([
    {'Check':'Starts and ends at depot','Pass':audit['closed_route_starts_and_ends_at_depot'],'Observed':'DEPOT to DEPOT'},
    {'Check':'Every delivery stop visited exactly once','Pass':audit['each_delivery_stop_visited_once'],'Observed':len(set(route[1:-1]))},
    {'Check':'All required stops included','Pass':audit['all_nodes_included'],'Observed':f"{len(route)-2} of {len(locations)-1}"},
    {'Check':'Distance matches independent leg sum','Pass':audit['distance_matches_optimizer'],'Observed':f"{audit['distance_recomputed_from_legs']:.6f} km"},
    {'Check':'ACO matches exact optimum on this instance','Pass':audit['gap_to_exact_percent'] < 1e-8,'Observed':f"{audit['gap_to_exact_percent']:.6f}% gap"},
])
assert audit_table['Pass'].all(), 'Route or distance audit failed.'
display(audit_table)


## 8. Monochrome convergence figure
The figure records the global best-so-far distance over iterations.

In [ ]:
if plt is not None:
    fig, ax = plt.subplots(figsize=(8,3.4))
    ax.plot(range(1,len(history)+1),history,color='black',linewidth=1.5)
    ax.set_title('ACO best-so-far route distance')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Distance (km)')
    ax.grid(True,color='0.85',linewidth=.5); fig.tight_layout(); plt.show()
else:
    print('Optional matplotlib is unavailable; the convergence table is complete.')


## 9. Export results
The CSV files are written into the Colab runtime. Open the Files panel to download them.

In [ ]:
benchmark_table.to_csv('route_benchmark_results.csv',index=False)
route_table.to_csv('optimized_route_legs.csv',index=False)
location_table.to_csv('synthetic_locations.csv',index=False)
distance_matrix_table.to_csv('pairwise_distance_matrix.csv')
convergence_table.to_csv('aco_convergence_summary.csv',index=False)
audit_table.to_csv('route_guardrail_audit.csv',index=False)
print('Exported six CSV result tables.')


## 10. Interpretation and limitations
On this toy map, the ACO + 2-opt route matches the exhaustive optimum at 31.154 km; nearest neighbor is 43.401 km and the 500-tour random mean is 53.455 km. This demonstrates the implementation on one small instance only. It does not establish performance on other maps. Straight-line distance omits road networks, direction restrictions, traffic, time windows, vehicle capacity, fuel use, delivery priorities, safety, and multiple vehicles. Exhaustive enumeration grows factorially and is included only as a reference for nine stops. Do not use this notebook for dispatch or live navigation.